In [6]:
import os
import glob
import json
import pdfplumber
from pptx import Presentation
from docx import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import ollama

# --- Configuration ---
INPUT_DIR = "./company_docs"
OUTPUT_FILE = "local_knowledge_base.jsonl"
EMBEDDING_MODEL = "mxbai-embed-large:latest"

# --- 1. Document Extraction Functions ---

def extract_pdf_with_tables(file_path):
    """Extracts text and converts tables to Markdown format from a PDF."""
    full_text = []
    print(f"Parsing PDF: {os.path.basename(file_path)}")
    
    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text() or ""
            full_text.append(f"\n--- Page {page_num} ---\n{page_text}")
            
            tables = page.extract_tables()
            for table in tables:
                if not table:
                    continue
                markdown_table = "\n[Table Extraction]\n"
                for row in table:
                    cleaned_row = [str(cell).replace("\n", " ").strip() if cell else "" for cell in row]
                    markdown_table += "| " + " | ".join(cleaned_row) + " |\n"
                full_text.append(markdown_table)
                
    return "\n".join(full_text)

def extract_pptx_with_tables(file_path):
    """Extracts text and table data from PowerPoint slides."""
    full_text = []
    print(f"Parsing PPTX: {os.path.basename(file_path)}")
    prs = Presentation(file_path)
    
    for slide_num, slide in enumerate(prs.slides, start=1):
        full_text.append(f"\n--- Slide {slide_num} ---\n")
        for shape in slide.shapes:
            if shape.has_text_frame:
                for paragraph in shape.text_frame.paragraphs:
                    if paragraph.text:
                        full_text.append(paragraph.text)
            
            if shape.has_table:
                markdown_table = "\n[Table Extraction]\n"
                for row in shape.table.rows:
                    cells = [cell.text.replace("\n", " ").strip() for cell in row.cells]
                    markdown_table += "| " + " | ".join(cells) + " |\n"
                full_text.append(markdown_table)
                
    return "\n".join(full_text)

def extract_docx_with_tables(file_path):
    """Extracts text and tables from Word documents."""
    full_text = []
    print(f"Parsing DOCX: {os.path.basename(file_path)}")
    doc = Document(file_path)
    
    # Extract standard paragraphs
    for para in doc.paragraphs:
        if para.text.strip():
            full_text.append(para.text.strip())
            
    # Extract tables and format as Markdown
    for table in doc.tables:
        markdown_table = "\n[Table Extraction]\n"
        for row in table.rows:
            # Clean newlines from cells so they stay on one row in Markdown
            cells = [cell.text.replace("\n", " ").strip() for cell in row.cells]
            markdown_table += "| " + " | ".join(cells) + " |\n"
        full_text.append(markdown_table)
        
    return "\n".join(full_text)

def extract_txt(file_path):
    """Extracts text from a plain text file."""
    print(f"Parsing TXT: {os.path.basename(file_path)}")
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

# --- 2. Main Orchestrator ---

def main():
    if not os.path.exists(INPUT_DIR):
        os.makedirs(INPUT_DIR)
        print(f"Created '{INPUT_DIR}' directory. Put your files there and re-run.")
        return

    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)

    # Now looking for .docx as well
    files = []
    for ext in ["*.pdf", "*.txt", "*.ppt", "*.pptx", "*.docx"]:
        files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))

    if not files:
        print(f"No documents found in '{INPUT_DIR}'.")
        return

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=50,
        length_function=len
    )

    print(f"Found {len(files)} file(s). Starting pipeline...\n")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_file:
        for file_path in files:
            ext = os.path.splitext(file_path)[1].lower()
            
            try:
                if ext == ".pdf":
                    text_content = extract_pdf_with_tables(file_path)
                elif ext in [".ppt", ".pptx"]:
                    text_content = extract_pptx_with_tables(file_path)
                elif ext == ".docx":
                    text_content = extract_docx_with_tables(file_path)
                elif ext == ".txt":
                    text_content = extract_txt(file_path)
                else:
                    continue
            except Exception as e:
                print(f"Skipping {file_path} due to parsing error: {e}")
                continue

            chunks = text_splitter.split_text(text_content)
            print(f"→ Split into {len(chunks)} chunks.")

            for idx, chunk in enumerate(chunks):
                try:
                    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=chunk)
                    embedding = response["embedding"]

                    payload = {
                        "source_file": os.path.basename(file_path),
                        "chunk_index": idx,
                        "text": chunk,
                        "embedding": embedding 
                    }
                    
                    out_file.write(json.dumps(payload) + "\n")
                except Exception as e:
                    print(f"Error embedding chunk {idx} of {file_path}: {e}")

    print(f"\n✅ Done! Data, tables, and vector arrays saved locally to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Found 2 file(s). Starting pipeline...

Parsing PPTX: 1. Introduction.pptx
→ Split into 27 chunks.
Parsing DOCX: 8 queens software technologies.docx
→ Split into 65 chunks.

✅ Done! Data, tables, and vector arrays saved locally to: local_knowledge_base.jsonl


In [1]:
import os
import json
import math
import ollama

# --- Configuration ---
KNOWLEDGE_BASE_FILE = "local_knowledge_base.jsonl"
EMBEDDING_MODEL = "mxbai-embed-large"
LLM_MODEL = "qwen3.5:0.8b"  # Or "llama3.2:3b"
TOP_K = 3                 # Number of context chunks to pass to the LLM

# --- Math Helpers for CPU Vector Search ---

def dot_product(vec1, vec2):
    return sum(a * b for a, b in zip(vec1, vec2))

def magnitude(vec):
    return math.sqrt(sum(a * a for a in vec))

def cosine_similarity(vec1, vec2):
    """Calculates the semantic closeness between two vector arrays."""
    mag1 = magnitude(vec1)
    mag2 = magnitude(vec2)
    if not mag1 or not mag2:
        return 0.0
    return dot_product(vec1, vec2) / (mag1 * mag2)

# --- Core RAG Logic ---

def load_knowledge_base():
    """Loads all embedded chunks from the local text file into memory."""
    if not os.path.exists(KNOWLEDGE_BASE_FILE):
        print(f"Error: Knowledge base file '{KNOWLEDGE_BASE_FILE}' not found.")
        print("Please run your ingestion script first to generate your data.")
        return []
    
    knowledge_base = []
    with open(KNOWLEDGE_BASE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                knowledge_base.append(json.loads(line))
    return knowledge_base

def retrieve_context(query, knowledge_base, top_k=3):
    """Embeds the query and scores it against all stored document chunks."""
    # 1. Convert user question into an embedding vector
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=query)
    query_embedding = response["embedding"]
    
    # 2. Calculate similarities across the knowledge base
    scored_chunks = []
    for chunk in knowledge_base:
        similarity = cosine_similarity(query_embedding, chunk["embedding"])
        scored_chunks.append((similarity, chunk))
    
    # 3. Sort by highest similarity score first
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    
    # Return the best matching chunks up to our Top_K limit
    return scored_chunks[:top_k]

def run_query_engine():
    print("Initializing local query engine...")
    knowledge_base = load_knowledge_base()
    if not knowledge_base:
        return
        
    print(f"Successfully loaded {len(knowledge_base)} document chunks into memory.")
    print(f"Using LLM: {LLM_MODEL} | Embedding: {EMBEDDING_MODEL}")
    print("-" * 60)
    print("System ready. Ask your question below (Type 'exit' or 'quit' to stop).\n")

    while True:
        user_query = input("\n🧑 Employee Question: ").strip()
        if not user_query:
            continue
        if user_query.lower() in ["exit", "quit"]:
            print("Shutting down query engine. Goodbye!")
            break
            
        print("\n🔍 Searching local files and computing math structures...")
        relevant_matches = retrieve_context(user_query, knowledge_base, top_k=TOP_K)
        
        if not relevant_matches or relevant_matches[0][0] < 0.3:
            print("⚠️ No highly relevant documentation context found for this request.")
            # We still proceed, but let the user know context is low
            
        # Compile the text sections to inject into the LLM prompt
        context_str = ""
        print("📄 Source Context Retrieved:")
        for idx, (score, chunk) in enumerate(relevant_matches, start=1):
            print(f"   [{idx}] {chunk['source_file']} (Confidence: {score:.2f})")
            context_str += f"\n--- Source: {chunk['source_file']} ---\n{chunk['text']}\n"
        print()

        # Construct a strict, context-grounded system prompt
        system_prompt = (
            "You are a secure corporate AI assistant. Answer the user's question accurately "
            "using ONLY the provided text context from company files. If the answer cannot be found "
            "or reasonably inferred from the context, state honestly that you do not know based on the "
            "available documents. Do not invent facts outside the text.\n\n"
            f"--- START COMPANY DOCUMENT CONTEXT ---\n{context_str}\n--- END COMPANY DOCUMENT CONTEXT ---"
        )

        print("🤖 Local AI Generating Response: ", end="", flush=True)
        
        try:
            # Stream the response word-by-word to bypass CPU latency perception
            stream = ollama.chat(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_query}
                ],
                stream=True,
            )
            
            for chunk in stream:
                print(chunk['message']['content'], end='', flush=True)
            print("\n" + "="*60)
            
        except Exception as e:
            print(f"\nAn error occurred during text generation: {e}")

if __name__ == "__main__":
    run_query_engine()

Initializing local query engine...
Successfully loaded 92 document chunks into memory.
Using LLM: qwen3.5:0.8b | Embedding: mxbai-embed-large
------------------------------------------------------------
System ready. Ask your question below (Type 'exit' or 'quit' to stop).


🔍 Searching local files and computing math structures...
📄 Source Context Retrieved:
   [1] 8 queens software technologies.docx (Confidence: 0.67)
   [2] 8 queens software technologies.docx (Confidence: 0.57)
   [3] 8 queens software technologies.docx (Confidence: 0.53)

🤖 Local AI Generating Response: 

KeyboardInterrupt: 

In [ ]:
import os
import json
import math
import time
import ollama

# --- Configuration ---
KNOWLEDGE_BASE_FILE = "local_knowledge_base.jsonl"
EMBEDDING_MODEL = "mxbai-embed-large"
LLM_MODEL = "qwen3.5:0.8b"  
TOP_K = 3                 

# --- Math Helpers for CPU Vector Search ---
def dot_product(vec1, vec2):
    return sum(a * b for a, b in zip(vec1, vec2))

def magnitude(vec):
    return math.sqrt(sum(a * a for a in vec))

def cosine_similarity(vec1, vec2):
    mag1 = magnitude(vec1)
    mag2 = magnitude(vec2)
    if not mag1 or not mag2:
        return 0.0
    return dot_product(vec1, vec2) / (mag1 * mag2)

# --- Core RAG Logic ---
def load_knowledge_base():
    if not os.path.exists(KNOWLEDGE_BASE_FILE):
        print(f"Error: Knowledge base file '{KNOWLEDGE_BASE_FILE}' not found.")
        return []
    
    knowledge_base = []
    with open(KNOWLEDGE_BASE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                knowledge_base.append(json.loads(line))
    return knowledge_base

def retrieve_context(query, knowledge_base, top_k=3):
    print("Loading embedding model and calculating vector...", end="", flush=True)
    start_time = time.time()
    
    # THE FIX: keep_alive=0 forces Ollama to drop the model from RAM immediately
    response = ollama.embeddings(
        model=EMBEDDING_MODEL, 
        prompt=query,
        keep_alive=0  
    )
    query_embedding = response["embedding"]
    
    print(f" Done! ({time.time() - start_time:.1f}s)")
    
    scored_chunks = []
    for chunk in knowledge_base:
        similarity = cosine_similarity(query_embedding, chunk["embedding"])
        scored_chunks.append((similarity, chunk))
    
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    return scored_chunks[:top_k]

def run_query_engine():
    knowledge_base = load_knowledge_base()
    if not knowledge_base:
        return
        
    print(f"Loaded {len(knowledge_base)} document chunks into memory.")
    print("-" * 60)

    while True:
        user_query = input("\nQuestion: ").strip()
        if not user_query:
            continue
        if user_query.lower() in ["exit", "quit"]:
            break
            
        print("\nStep 1: Searching local files...")
        relevant_matches = retrieve_context(user_query, knowledge_base, top_k=TOP_K)
        
        context_str = ""
        for idx, (score, chunk) in enumerate(relevant_matches, start=1):
            context_str += f"\n--- Source: {chunk['source_file']} ---\n{chunk['text']}\n"

        system_prompt = (
            "You are a secure corporate AI assistant. Answer the user's question accurately "
            "using ONLY the provided text context from company files. Do not invent facts.\n\n"
            f"--- START CONTEXT ---\n{context_str}\n--- END CONTEXT ---"
        )

        print(f"\nStep 2: Loading {LLM_MODEL} into RAM and generating text...")
        print("AI: ", end="", flush=True)
        
        try:
            start_gen_time = time.time()
            first_token_printed = False
            
            stream = ollama.chat(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_query}
                ],
                stream=True,
            )
            
            for chunk in stream:
                if not first_token_printed:
                    # This tells us exactly how long it took to load the LLM
                    print(f"\n[Started typing after {time.time() - start_gen_time:.1f}s] ", end="")
                    first_token_printed = True
                    
                print(chunk['message']['content'], end='', flush=True)
                
            print("\n" + "="*60)
            
        except Exception as e:
            print(f"\nError: {e}")

if __name__ == "__main__":
    run_query_engine()

✅ Loaded 92 document chunks into memory.
------------------------------------------------------------

🔍 Step 1: Searching local files...
   -> Loading embedding model and calculating vector... Done! (1.1s)

🧠 Step 2: Loading qwen3.5:0.8b into RAM and generating text...
🤖 AI: 
[Started typing after 0.3s] Based on the provided text from the 8 queens software technologies.docx, the company provides the following services:

*   Web design
*   Web application
*   Digital marketing
*   Customized Software
*   Mobile Application
*   Design
*   Development & Implementation
*   Upgrade & Modernization
*   Support and Maintenance
*   Analytics and Reporting
*   Integration
*   Mobility
